# Phase II: Time-Conditioned Latent Diffusion

**Pipeline**
1. Frozen Phase I VAE encodes the source slice `x_t` → `z_src` (B, 64, 32, 32)
   and the follow-up slice `x_{t+Δt}` → `z_tgt` (B, 64, 32, 32).
2. Both latents are rescaled by a fixed factor `LATENT_SCALE` so they sit at
   roughly unit variance (Stable-Diffusion-style 0.18215, but computed
   empirically from training data here since our β-VAE was trained with low
   KL weight).
3. The diffusion U-Net learns ε(z_tau, τ, Δt, z_src) by DDPM noise-prediction
   in the latent space.
4. Cross-attention context = `z_src` reshaped to 1024 tokens of dim 64.
5. Conditioning: τ and Δt each get a sinusoidal embedding → MLP → FiLM
   modulation injected at every ResBlock.
6. At inference: DDIM-sample `z_pred` (50 steps) → unscale →
   `vae.decode(z_pred)` → pixel-space follow-up prediction.


## 1. Setup


In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


In [2]:
get_ipython().system('nvidia-smi')
get_ipython().system('pip install -q torch torchvision nibabel scikit-image lpips tqdm pandas matplotlib')


Fri May 15 18:28:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   31C    P0             47W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [3]:
import os, json, math, time, random
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)


Device: cuda


## 2. Configuration


In [4]:
PROJECT_DIR   = Path('/content/drive/MyDrive/project')
MANIFEST_CSV  = PROJECT_DIR / 'manifests' / 'slices_manifest_full_300.csv'
VAE_CKPT      = PROJECT_DIR / 'chronodiff' / 'phase1_vae' / 'new_vae_phase1_150.pt'
OUT_DIR       = PROJECT_DIR / 'chronodiff' / 'phase_2_300'
OUT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_PATH     = OUT_DIR / 'chronodiff_phase2_300.pt'
LOG_PATH      = OUT_DIR / 'train_log_300.json'
PRED_CSV      = OUT_DIR / 'chronodiff_full_per_slice.csv'

IMG_SIZE      = 256          # matches Phase I VAE input
LATENT_CH     = 64           # matches Phase I VAE latent channels
LATENT_SIZE   = 32
VAE_BASE_CH   = 32           # matches Phase I


BASE_CH       = 128          # channel base for the latent U-Net
TIME_EMB_DIM  = 128
T_DIFFUSION   = 1000
DDIM_STEPS    = 50

CFG_DROP_PROB = 0.10
CFG_SCALE     = 2.0

BATCH_SIZE    = 16           # latent diffusion is cheap → larger batch is fine
N_EPOCHS      = 60           # v3: doubled from 30. Val curve was still flat at
LR            = 1e-4
NUM_WORKERS   = 0
FORCE_RETRAIN = True        # if True, ignore existing CKPT_PATH

print(f'Manifest:   {MANIFEST_CSV}')
print(f'VAE ckpt:   {VAE_CKPT}')
print(f'Out dir:    {OUT_DIR}')
print(f'CKPT path:  {CKPT_PATH}')
print(f'Per-slice:  {PRED_CSV}')
print(f'Config: IMG={IMG_SIZE} LATENT={LATENT_CH}x{LATENT_SIZE}x{LATENT_SIZE} '
      f'BASE_CH={BASE_CH} BS={BATCH_SIZE} EPOCHS={N_EPOCHS}')

assert MANIFEST_CSV.exists(),  f'Missing manifest: {MANIFEST_CSV}'
assert VAE_CKPT.exists(),      f'Missing VAE ckpt: {VAE_CKPT}'

Manifest:   /content/drive/MyDrive/project/manifests/slices_manifest_full_300.csv
VAE ckpt:   /content/drive/MyDrive/project/chronodiff/phase1_vae/new_vae_phase1_150.pt
Out dir:    /content/drive/MyDrive/project/chronodiff/phase_2_300
CKPT path:  /content/drive/MyDrive/project/chronodiff/phase_2_300/chronodiff_phase2_300.pt
Per-slice:  /content/drive/MyDrive/project/chronodiff/phase_2_300/chronodiff_full_per_slice.csv
Config: IMG=256 LATENT=64x32x32 BASE_CH=128 BS=16 EPOCHS=60


## 3. Load frozen Phase I VAE


In [5]:
class ConditionalVAE(nn.Module):
    """Phase I spatial-latent VAE — encoder outputs (B, latent_ch, 32, 32)."""
    def __init__(self, latent_ch: int = 64, base_ch: int = 32):
        super().__init__()
        self.latent_ch = latent_ch
        self.enc = nn.Sequential(
            nn.Conv2d(1,          base_ch,     4, stride=2, padding=1),
            nn.GroupNorm(8, base_ch),     nn.SiLU(),
            nn.Conv2d(base_ch,    base_ch * 2, 4, stride=2, padding=1),
            nn.GroupNorm(8, base_ch * 2), nn.SiLU(),
            nn.Conv2d(base_ch * 2, base_ch * 4, 4, stride=2, padding=1),
            nn.GroupNorm(8, base_ch * 4), nn.SiLU(),
        )
        self.to_latent = nn.Conv2d(base_ch * 4, 2 * latent_ch, kernel_size=1)
        self.from_latent = nn.Conv2d(latent_ch, base_ch * 4, kernel_size=1)
        self.dec = nn.Sequential(
            nn.SiLU(),
            nn.ConvTranspose2d(base_ch * 4, base_ch * 2, 4, stride=2, padding=1),
            nn.GroupNorm(8, base_ch * 2), nn.SiLU(),
            nn.ConvTranspose2d(base_ch * 2, base_ch,     4, stride=2, padding=1),
            nn.GroupNorm(8, base_ch),     nn.SiLU(),
            nn.ConvTranspose2d(base_ch,     1,           4, stride=2, padding=1),
            nn.Sigmoid(),
        )

    def encode(self, x):
        h = self.enc(x)
        h = self.to_latent(h)
        mu, logvar = h.chunk(2, dim=1)
        return mu, logvar

    def decode(self, z):
        return self.dec(self.from_latent(z))

    def forward(self, x):
        mu, logvar = self.encode(x)
        std = (0.5 * logvar).exp()
        z = mu + std * torch.randn_like(std)
        return self.decode(z), mu, logvar

vae = ConditionalVAE(latent_ch=LATENT_CH, base_ch=VAE_BASE_CH).to(DEVICE)
state = torch.load(VAE_CKPT, map_location=DEVICE)
vae.load_state_dict(state['model'])
vae.eval()
for p in vae.parameters():
    p.requires_grad_(False)
print(f'Loaded Phase I VAE — epoch {state.get("epoch", "?")} '
      f'val={state.get("val_loss", float("nan")):.4f}')


Loaded Phase I VAE — epoch 35 val=0.0000


In [6]:
import torch
with torch.no_grad():
    x192 = torch.randn(1, 1, 192, 192).to(DEVICE)
    x256 = torch.randn(1, 1, 256, 256).to(DEVICE)
    z192 = vae.encode(x192)[0]
    z256 = vae.encode(x256)[0]
    print('VAE on raw 192:  ', tuple(z192.shape))  # expect (1, 64, 24, 24)
    print('VAE on resized 256:', tuple(z256.shape))  # expect (1, 64, 32, 32)

VAE on raw 192:   (1, 64, 24, 24)
VAE on resized 256: (1, 64, 32, 32)


## 4. Manifest, deterministic splits, paired dataset



In [7]:
manifest = pd.read_csv(MANIFEST_CSV)
print('Pair manifest rows:', len(manifest))

patients = sorted(manifest['patient_id'].unique())
rng = np.random.default_rng(SEED)
rng.shuffle(patients)
n = len(patients)
n_train = int(0.70 * n); n_val = int(0.15 * n)
train_p = set(patients[:n_train])
val_p   = set(patients[n_train:n_train + n_val])
test_p  = set(patients[n_train + n_val:])

def split_df(df, pset):
    return df[df['patient_id'].isin(pset)].reset_index(drop=True)

train_df = split_df(manifest, train_p)
val_df   = split_df(manifest, val_p)
test_df  = split_df(manifest, test_p)
print(f'Patients: tr={len(train_p)} va={len(val_p)} te={len(test_p)}')
print(f'Pairs:    tr={len(train_df)} va={len(val_df)} te={len(test_df)}')

# --- ONE source of truth for Δt normalisation ---
DT_NORM = float(np.percentile(train_df['delta_t_days'].values, 95))
print(f'DT_NORM (95th percentile of train Δt): {DT_NORM:.1f} days')


Pair manifest rows: 152866
Patients: tr=210 va=45 te=45
Pairs:    tr=109896 va=21017 te=21953
DT_NORM (95th percentile of train Δt): 339.0 days


In [8]:
class PairDataset(Dataset):
    """Returns (x_t, x_next, dt_norm) for ChronoDiff Phase II training."""
    def __init__(self, df, img_size=IMG_SIZE, dt_norm=DT_NORM, augment=False):
        self.df = df.reset_index(drop=True)
        self.img_size = img_size
        self.dt_norm = dt_norm
        self.augment = augment

    def __len__(self):
        return len(self.df)

    def _load(self, p):
        a = np.load(p).astype(np.float32)
        a = np.clip(a, -3.0, 3.0)
        a = (a + 3.0) / 6.0                  # → [0, 1] to match Phase I
        t = torch.from_numpy(a).unsqueeze(0)  # (1, H, W)
        if t.shape[-1] != self.img_size:
            t = TF.resize(t, [self.img_size, self.img_size], antialias=True)
        return t

    def __getitem__(self, i):
        r = self.df.iloc[i]
        x_t = self._load(r['slice_t'])
        x_n = self._load(r['slice_next'])
        if self.augment and random.random() < 0.5:    # horizontal flip
            x_t = torch.flip(x_t, [-1])
            x_n = torch.flip(x_n, [-1])
        dt = torch.tensor(float(r['delta_t_days']) / self.dt_norm,
                          dtype=torch.float32).clamp(0.0, 3.0)
        # slice_id is needed for the per-slice CSV
        sid = str(r.get('slice_id', f'{r["patient_id"]}_{i}'))
        return x_t, x_n, dt, sid

def _collate(batch):
    xt = torch.stack([b[0] for b in batch], 0)
    xn = torch.stack([b[1] for b in batch], 0)
    dt = torch.stack([b[2] for b in batch], 0)
    sids = [b[3] for b in batch]
    return xt, xn, dt, sids

train_ds = PairDataset(train_df, augment=True)
val_ds   = PairDataset(val_df,   augment=False)
test_ds  = PairDataset(test_df,  augment=False)

dl_kw = dict(num_workers=NUM_WORKERS, pin_memory=True,
             persistent_workers=(NUM_WORKERS > 0), collate_fn=_collate)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          drop_last=True, **dl_kw)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, **dl_kw)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, **dl_kw)

xt, xn, dt, sids = next(iter(train_loader))
print(f'x_t={tuple(xt.shape)} range=[{xt.min():.3f},{xt.max():.3f}] '
      f'dt range=[{dt.min():.3f},{dt.max():.3f}] (norm by {DT_NORM:.0f}d)')


x_t=(16, 1, 256, 256) range=[0.000,1.000] dt range=[0.083,0.968] (norm by 339d)


## 5. Compute latent scale factor



In [9]:
@torch.no_grad()
def estimate_latent_scale(loader, n_batches=20):
    stds = []
    for i, (xt, xn, dt, _) in enumerate(loader):
        if i >= n_batches: break
        z, _ = vae.encode(xt.to(DEVICE))
        stds.append(z.float().std().item())
    return float(np.mean(stds))

raw_std = estimate_latent_scale(train_loader)
LATENT_SCALE = 1.0 / max(raw_std, 1e-6)
print(f'Empirical latent std on train: {raw_std:.4f}')
print(f'LATENT_SCALE (1 / std):        {LATENT_SCALE:.4f}')

@torch.no_grad()
def encode_pair(x_t, x_n):
    """Encode source + target slices to standardized latents."""
    z_src = vae.encode(x_t)[0] * LATENT_SCALE   # (B, 64, 32, 32)
    z_tgt = vae.encode(x_n)[0] * LATENT_SCALE
    return z_src, z_tgt

@torch.no_grad()
def decode_latent(z):
    return vae.decode(z / LATENT_SCALE)         # (B, 1, 256, 256), in [0,1]


Empirical latent std on train: 0.4477
LATENT_SCALE (1 / std):        2.2336


## 6. Diffusion model components


In [10]:
class SinusoidalEmbedding(nn.Module):
    def __init__(self, dim, max_period=10000):
        super().__init__(); self.dim = dim; self.max_period = max_period
    def forward(self, x):
        x = x.float()
        half = self.dim // 2
        freqs = torch.exp(-math.log(self.max_period) *
                          torch.arange(half, device=x.device).float() / half)
        args = x[:, None] * freqs[None, :]
        emb = torch.cat([torch.cos(args), torch.sin(args)], dim=-1)
        if self.dim % 2:
            emb = F.pad(emb, (0, 1))
        return emb

class FiLM(nn.Module):
    def __init__(self, cond_dim, channels):
        super().__init__()
        self.to_gb = nn.Linear(cond_dim, channels * 2)
    def forward(self, x, c):
        gb = self.to_gb(c)
        g, b = gb.chunk(2, dim=-1)
        g = torch.tanh(g) * 0.5
        return x * (1 + g[:, :, None, None]) + b[:, :, None, None]

class CrossAttention(nn.Module):
    def __init__(self, channels, ctx_dim, heads=4):
        super().__init__()
        assert channels % heads == 0
        self.heads = heads
        self.norm = nn.GroupNorm(8, channels)
        self.to_q = nn.Conv2d(channels, channels, 1)
        self.to_k = nn.Linear(ctx_dim, channels)
        self.to_v = nn.Linear(ctx_dim, channels)
        self.to_out = nn.Conv2d(channels, channels, 1)
        # Fix (A): zero-init the residual output projection so attention starts
        # as an identity contribution.
        nn.init.zeros_(self.to_out.weight)
        nn.init.zeros_(self.to_out.bias)
    def forward(self, x, ctx):
        B, C, H, W = x.shape
        N = ctx.size(1)
        d = C // self.heads
        q = self.to_q(self.norm(x)).view(B, self.heads, d, H * W)         # (B,h,d,HW)
        k = self.to_k(ctx).view(B, N, self.heads, d).permute(0, 2, 3, 1)  # (B,h,d,N)
        v = self.to_v(ctx).view(B, N, self.heads, d).permute(0, 2, 3, 1)  # (B,h,d,N)
        attn = torch.softmax((q.transpose(-1, -2) @ k) / math.sqrt(d), dim=-1)  # (B,h,HW,N)
        out = (attn @ v.transpose(-1, -2)).transpose(-1, -2)              # (B,h,d,HW)
        out = out.reshape(B, C, H, W)
        return x + self.to_out(out)

class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, cond_dim, ctx_dim, with_attn=True):
        super().__init__()
        self.norm1 = nn.GroupNorm(8, in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.film  = FiLM(cond_dim, out_ch)
        self.norm2 = nn.GroupNorm(8, out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.attn  = CrossAttention(out_ch, ctx_dim) if with_attn else None
        self.skip  = nn.Identity() if in_ch == out_ch else nn.Conv2d(in_ch, out_ch, 1)
    def forward(self, x, cond, ctx):
        h = self.conv1(F.silu(self.norm1(x)))
        h = self.film(h, cond)
        h = self.conv2(F.silu(self.norm2(h)))
        if self.attn is not None:
            h = self.attn(h, ctx)
        return h + self.skip(x)

class Down(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.op = nn.Conv2d(ch, ch, 3, stride=2, padding=1)
    def forward(self, x): return self.op(x)

class Up(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.op = nn.ConvTranspose2d(ch, ch, 4, stride=2, padding=1)
    def forward(self, x): return self.op(x)


## 7. Latent-space time-conditioned U-Net


In [11]:
class LatentTimeUNet(nn.Module):
    def __init__(self, latent_ch=LATENT_CH, base_ch=BASE_CH,
                 time_emb_dim=TIME_EMB_DIM, ctx_dim=LATENT_CH,
                 ctx_tokens=LATENT_SIZE * LATENT_SIZE):
        super().__init__()
        self.tau_emb = nn.Sequential(
            SinusoidalEmbedding(time_emb_dim),
            nn.Linear(time_emb_dim, time_emb_dim), nn.SiLU(),
            nn.Linear(time_emb_dim, time_emb_dim))
        self.dt_emb = nn.Sequential(
            SinusoidalEmbedding(time_emb_dim),
            nn.Linear(time_emb_dim, time_emb_dim), nn.SiLU(),
            nn.Linear(time_emb_dim, time_emb_dim))
        self.cond_proj = nn.Sequential(
            nn.Linear(time_emb_dim * 2, time_emb_dim), nn.SiLU())

        self.ctx_pos_emb = nn.Parameter(torch.zeros(1, ctx_tokens, ctx_dim))
        nn.init.normal_(self.ctx_pos_emb, std=0.02)

        c = base_ch
        self.in_conv = nn.Conv2d(2 * latent_ch, c, 3, padding=1)

        # Encoder: 32 -> 16 -> 8
        self.d1 = ResBlock(c,     c,     time_emb_dim, ctx_dim); self.p1 = Down(c)
        self.d2 = ResBlock(c,     2 * c, time_emb_dim, ctx_dim); self.p2 = Down(2 * c)
        self.d3 = ResBlock(2 * c, 2 * c, time_emb_dim, ctx_dim)

        # Bottleneck at 8x8
        self.mid1 = ResBlock(2 * c, 4 * c, time_emb_dim, ctx_dim)
        self.mid2 = ResBlock(4 * c, 4 * c, time_emb_dim, ctx_dim)

        # Decoder: 8 -> 16 -> 32 (also one in-place block at 8x8 after mid)
        self.u2 = Up(4 * c)
        self.r2 = ResBlock(4 * c + 2 * c, 2 * c, time_emb_dim, ctx_dim)
        self.u1 = Up(2 * c)
        self.r1 = ResBlock(2 * c + c, c, time_emb_dim, ctx_dim)

        self.out = nn.Sequential(
            nn.GroupNorm(8, c), nn.SiLU(),
            nn.Conv2d(c, latent_ch, 3, padding=1))

    def forward(self, z_tau, tau, dt, ctx, z_src):
        tau_e = self.tau_emb(tau.float())
        dt_e  = self.dt_emb(dt.float())
        cond  = self.cond_proj(torch.cat([tau_e, dt_e], dim=-1))

        ctx = ctx + self.ctx_pos_emb
        z_in = torch.cat([z_tau, z_src], dim=1)              # (B, 2*latent_ch, 32, 32)
        h0 = self.in_conv(z_in)
        h1 = self.d1(h0, cond, ctx); p1 = self.p1(h1)       # 32 -> 16
        h2 = self.d2(p1, cond, ctx); p2 = self.p2(h2)       # 16 -> 8
        h3 = self.d3(p2, cond, ctx)                          # 8

        m = self.mid1(h3, cond, ctx)
        m = self.mid2(m,  cond, ctx)

        u2 = self.u2(m);  u2 = self.r2(torch.cat([u2, h2], 1), cond, ctx)   # 8 -> 16
        u1 = self.u1(u2); u1 = self.r1(torch.cat([u1, h1], 1), cond, ctx)   # 16 -> 32
        return self.out(u1)

unet = LatentTimeUNet().to(DEVICE)
n_params = sum(p.numel() for p in unet.parameters())
print(f'Latent U-Net params: {n_params / 1e6:.2f} M')

with torch.no_grad():
    z_src, z_tgt = encode_pair(xt[:2].to(DEVICE), xn[:2].to(DEVICE))
    B, C, H, W = z_src.shape
    ctx = z_src.reshape(B, C, H * W).transpose(1, 2)         # (B, 1024, 64)
    tau = torch.randint(0, T_DIFFUSION, (B,), device=DEVICE)
    dt_b = dt[:2].to(DEVICE)
    eps_hat = unet(z_tgt, tau, dt_b, ctx, z_src)
    print(f'  z_src:   {tuple(z_src.shape)}')
    print(f'  z_tgt:   {tuple(z_tgt.shape)}')
    print(f'  ctx:     {tuple(ctx.shape)}')
    print(f'  ε̂ shape: {tuple(eps_hat.shape)}')
    assert eps_hat.shape == z_tgt.shape, 'ε̂ shape mismatch!'


Latent U-Net params: 22.67 M
  z_src:   (2, 64, 32, 32)
  z_tgt:   (2, 64, 32, 32)
  ctx:     (2, 1024, 64)
  ε̂ shape: (2, 64, 32, 32)


## 8. Diffusion schedule (DDPM forward + DDIM inverse + CFG)


In [12]:
class LatentDiffusion:
    def __init__(self, T=T_DIFFUSION, beta_start=1e-4, beta_end=0.02, device=DEVICE):
        self.T = T; self.device = device
        self.betas       = torch.linspace(beta_start, beta_end, T, device=device)
        self.alphas      = 1.0 - self.betas
        self.alpha_bars  = torch.cumprod(self.alphas, dim=0)

    def q_sample(self, z0, tau, noise):
        a_bar = self.alpha_bars[tau.long()].view(-1, 1, 1, 1)
        return torch.sqrt(a_bar) * z0 + torch.sqrt(1 - a_bar) * noise

    @torch.no_grad()
    def ddim_sample(self, unet, z_src, dt, n_steps=DDIM_STEPS,
                    cfg_scale=CFG_SCALE):
        B = z_src.size(0)
        z = torch.randn(B, LATENT_CH, LATENT_SIZE, LATENT_SIZE, device=self.device)
        ctx       = z_src.reshape(B, LATENT_CH, LATENT_SIZE * LATENT_SIZE).transpose(1, 2)
        ctx_null  = torch.zeros_like(ctx)
        dt_null   = torch.zeros_like(dt)
        ts = torch.linspace(self.T - 1, 0, n_steps + 1, device=self.device).long()
        for i in range(n_steps):
            t_cur = ts[i].expand(B);     t_nxt = ts[i + 1].expand(B)
            ab_cur = self.alpha_bars[t_cur].view(-1, 1, 1, 1)
            ab_nxt = self.alpha_bars[t_nxt].view(-1, 1, 1, 1)
            if cfg_scale == 1.0:
                eps = unet(z, t_cur, dt, ctx, z_src)
            else:
                eps_cond   = unet(z, t_cur, dt,      ctx,      z_src)
                eps_uncond = unet(z, t_cur, dt_null, ctx_null, z_src)
                eps        = eps_uncond + cfg_scale * (eps_cond - eps_uncond)
            z0  = (z - torch.sqrt(1 - ab_cur) * eps) / torch.sqrt(ab_cur)
            z   = torch.sqrt(ab_nxt) * z0 + torch.sqrt(1 - ab_nxt) * eps
        return z0   # the clean latent prediction at the final step

    @torch.no_grad()
    def ddim_sample_sdedit(self, unet, z_src, dt, t_start_frac=0.4,
                           n_steps=DDIM_STEPS, cfg_scale=CFG_SCALE):
        B = z_src.size(0)
        ctx       = z_src.reshape(B, LATENT_CH, LATENT_SIZE * LATENT_SIZE).transpose(1, 2)
        ctx_null  = torch.zeros_like(ctx)
        dt_null   = torch.zeros_like(dt)

        # Forward-diffuse z_src to t_start instead of starting from pure noise.
        t_start  = int(t_start_frac * self.T)
        ab_start = self.alpha_bars[t_start]
        noise    = torch.randn_like(z_src)
        z        = torch.sqrt(ab_start) * z_src + torch.sqrt(1 - ab_start) * noise

        ts = torch.linspace(t_start, 0, n_steps + 1, device=self.device).long()
        for i in range(n_steps):
            t_cur = ts[i].expand(B);     t_nxt = ts[i + 1].expand(B)
            ab_cur = self.alpha_bars[t_cur].view(-1, 1, 1, 1)
            ab_nxt = self.alpha_bars[t_nxt].view(-1, 1, 1, 1)
            if cfg_scale == 1.0:
                eps = unet(z, t_cur, dt, ctx, z_src)
            else:
                eps_cond   = unet(z, t_cur, dt,      ctx,      z_src)
                eps_uncond = unet(z, t_cur, dt_null, ctx_null, z_src)
                eps        = eps_uncond + cfg_scale * (eps_cond - eps_uncond)
            z0  = (z - torch.sqrt(1 - ab_cur) * eps) / torch.sqrt(ab_cur)
            z   = torch.sqrt(ab_nxt) * z0 + torch.sqrt(1 - ab_nxt) * eps
        return z0

diffusion = LatentDiffusion()
print(f'Diffusion: T={T_DIFFUSION}, DDIM steps={DDIM_STEPS}')


Diffusion: T=1000, DDIM steps=50


## 9. Training loop (ε-prediction, mixed precision)


In [ ]:
opt    = torch.optim.AdamW(unet.parameters(), lr=LR, weight_decay=1e-5)
sched  = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=N_EPOCHS)
scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE.type == 'cuda'))

# Resume guard — don't blow away a finished training run on a re-execute.
start_epoch = 1
history = {'epoch': [], 'train_loss': [], 'val_loss': [], 'lr': []}
best_val = float('inf')

if CKPT_PATH.exists() and not FORCE_RETRAIN:
    print(f'Found existing checkpoint at {CKPT_PATH} — set FORCE_RETRAIN=True '
          f'to overwrite. Loading and skipping training.')
    ck = torch.load(CKPT_PATH, map_location=DEVICE)
    unet.load_state_dict(ck['model'])
    start_epoch = N_EPOCHS + 1
    if LOG_PATH.exists():
        with open(LOG_PATH) as f: history = json.load(f)
    best_val = ck.get('val_loss', float('inf'))
    if 'latent_scale' in ck:
        LATENT_SCALE = float(ck['latent_scale'])
        print(f'  ↳ restored LATENT_SCALE from checkpoint: {LATENT_SCALE:.4f}')
    if 'dt_norm' in ck:
        DT_NORM = float(ck['dt_norm'])
        print(f'  ↳ restored DT_NORM from checkpoint:      {DT_NORM:.1f} days')
    cfg_ck = ck.get('config', {})
    if cfg_ck:
        assert cfg_ck.get('latent_size', LATENT_SIZE) == LATENT_SIZE, (
            f'Checkpoint latent_size={cfg_ck.get("latent_size")} != '
            f'current LATENT_SIZE={LATENT_SIZE}. Set FORCE_RETRAIN=True.')
        assert cfg_ck.get('latent_ch',   LATENT_CH)   == LATENT_CH

def run_epoch(loader, train=True):
    unet.train(train)
    tot, n = 0.0, 0
    pbar = tqdm(loader, desc='train' if train else 'val', leave=False)
    for x_t, x_n, dt, _sids in pbar:
        x_t = x_t.to(DEVICE, non_blocking=True)
        x_n = x_n.to(DEVICE, non_blocking=True)
        dt  = dt.to(DEVICE, non_blocking=True)
        B   = x_t.size(0)
        z_src, z_tgt = encode_pair(x_t, x_n)
        ctx = z_src.reshape(B, LATENT_CH, LATENT_SIZE * LATENT_SIZE).transpose(1, 2)
        tau = torch.randint(0, diffusion.T, (B,), device=DEVICE)
        noise = torch.randn_like(z_tgt)
        z_tau = diffusion.q_sample(z_tgt, tau, noise)

        if train:
            drop = (torch.rand(B, device=DEVICE) < CFG_DROP_PROB)
            keep = (~drop).float()
            ctx  = ctx * keep.view(B, 1, 1)
            dt   = dt  * keep                                # null Δt → 0
        with torch.amp.autocast('cuda', enabled=(DEVICE.type == 'cuda')):
            eps_pred = unet(z_tau, tau, dt, ctx, z_src)
            loss = F.mse_loss(eps_pred, noise)
        if train:
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(unet.parameters(), 1.0)
            scaler.step(opt); scaler.update()
        tot += loss.item() * B; n += B
        pbar.set_postfix(loss=f'{loss.item():.4f}')
    return tot / max(n, 1)

for ep in range(start_epoch, N_EPOCHS + 1):
    t0 = time.time()
    tr = run_epoch(train_loader, train=True)
    with torch.no_grad():
        va = run_epoch(val_loader, train=False)
    sched.step()
    lr_now = opt.param_groups[0]['lr']
    dt_s = time.time() - t0
    print(f'Epoch {ep:02d}/{N_EPOCHS} ({dt_s:.0f}s) | '
          f'train ε-MSE {tr:.4f} | val ε-MSE {va:.4f} | lr {lr_now:.2e}')
    history['epoch'].append(ep); history['train_loss'].append(tr)
    history['val_loss'].append(va); history['lr'].append(lr_now)
    if va < best_val - 1e-5:
        best_val = va
        torch.save({'model': unet.state_dict(),
                    'epoch': ep,
                    'val_loss': best_val,
                    'latent_scale': LATENT_SCALE,
                    'dt_norm': DT_NORM,
                    'config': {'base_ch': BASE_CH,
                               'time_emb_dim': TIME_EMB_DIM,
                               'latent_ch': LATENT_CH,
                               'latent_size': LATENT_SIZE,
                               'T_diffusion': T_DIFFUSION}},
                   CKPT_PATH)
        print(f'  ↳ saved {CKPT_PATH} (val={best_val:.4f})')
    with open(LOG_PATH, 'w') as f:
        json.dump(history, f, indent=2)


train:   0%|          | 0/6868 [00:00<?, ?it/s]

## 10. Training curves


In [ ]:
with open(LOG_PATH) as f: history = json.load(f)
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(history['epoch'], history['train_loss'], label='train')
ax[0].plot(history['epoch'], history['val_loss'],   label='val')
ax[0].set_title('ε-MSE'); ax[0].set_xlabel('epoch'); ax[0].legend()
ax[1].plot(history['epoch'], history['lr']); ax[1].set_title('LR')
ax[1].set_xlabel('epoch'); plt.tight_layout()
plt.savefig(OUT_DIR / 'training_curves.png', dpi=120); plt.show()


## 11. Qualitative samples
Sample with DDIM (50 steps), decode latents back to pixel space, and grid
baseline / GT follow-up / prediction.


In [ ]:
unet.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE)['model'])
unet.eval()

x_t, x_n, dt_b, _sids = next(iter(val_loader))
x_t = x_t[:6].to(DEVICE); x_n = x_n[:6].to(DEVICE); dt_b = dt_b[:6].to(DEVICE)
with torch.no_grad():
    z_src, _ = encode_pair(x_t, x_n)
    z_pred   = diffusion.ddim_sample(unet, z_src, dt_b, n_steps=DDIM_STEPS)
    x_pred   = decode_latent(z_pred).clamp(0, 1).cpu()

fig, ax = plt.subplots(3, 6, figsize=(18, 9))
for i in range(6):
    ax[0, i].imshow(x_t[i, 0].cpu(),  cmap='gray'); ax[0, i].axis('off')
    ax[1, i].imshow(x_n[i, 0].cpu(),  cmap='gray'); ax[1, i].axis('off')
    ax[2, i].imshow(x_pred[i, 0],     cmap='gray'); ax[2, i].axis('off')
ax[0, 0].set_title('Baseline x_t',         loc='left')
ax[1, 0].set_title('GT follow-up',         loc='left')
ax[2, 0].set_title('ChronoDiff prediction',loc='left')
plt.tight_layout(); plt.savefig(OUT_DIR / 'qualitative.png', dpi=120); plt.show()


## 12. Test-set evaluation + per-slice CSV


In [ ]:
pip install lpips

In [ ]:
from skimage.metrics import peak_signal_noise_ratio as psnr_fn
from skimage.metrics import structural_similarity as ssim_fn
import lpips as _lpips
lp_net = _lpips.LPIPS(net='alex').to(DEVICE).eval()
for p in lp_net.parameters(): p.requires_grad_(False)

rows = []
for x_t, x_n, dt_b, sids in tqdm(test_loader, desc='test'):
    x_t = x_t.to(DEVICE); x_n = x_n.to(DEVICE); dt_b = dt_b.to(DEVICE)
    with torch.no_grad():
        z_src, _ = encode_pair(x_t, x_n)
        z_pred   = diffusion.ddim_sample(unet, z_src, dt_b, n_steps=DDIM_STEPS)
        x_pred   = decode_latent(z_pred).clamp(0, 1)
        # LPIPS expects 3-ch in [-1, 1]
        lp = lp_net(x_pred.repeat(1, 3, 1, 1) * 2 - 1,
                    x_n.repeat(1, 3, 1, 1) * 2 - 1).squeeze().cpu().numpy()
    a = x_n.cpu().numpy(); b = x_pred.cpu().numpy()
    lp = np.atleast_1d(lp)
    for i in range(a.shape[0]):
        rows.append({
            'slice_id': sids[i],
            'mae':   float(np.mean(np.abs(a[i, 0] - b[i, 0]))),
            'mse':   float(np.mean((a[i, 0] - b[i, 0]) ** 2)),
            'psnr':  float(psnr_fn(a[i, 0], b[i, 0], data_range=1.0)),
            'ssim':  float(ssim_fn(a[i, 0], b[i, 0], data_range=1.0)),
            'lpips': float(lp[i]),
        })

df_pred = pd.DataFrame(rows)
df_pred.to_csv(PRED_CSV, index=False)
print(f'Wrote {len(df_pred)} per-slice rows to {PRED_CSV}')
print(df_pred[['mae', 'mse', 'psnr', 'ssim', 'lpips']].mean().to_string())


In [ ]:
# CFG-collapse diagnostic (Definition 1).
from skimage.metrics import structural_similarity as ssim_fn
from itertools import combinations

CFG_SWEEP   = [0.0, 1.0, 2.0]
N_CASES     = 64       # how many held-out slices to test on
EPSILON     = 1e-3     # collapse threshold

unet.eval()
xs = []
for x_t, x_n, dt_b, _ in test_loader:
    xs.append((x_t, x_n, dt_b))
    if sum(b[0].size(0) for b in xs) >= N_CASES: break
x_t = torch.cat([b[0] for b in xs], 0)[:N_CASES].to(DEVICE)
dt_b = torch.cat([b[2] for b in xs], 0)[:N_CASES].to(DEVICE)

per_w = {}
with torch.no_grad():
    z_src, _ = encode_pair(x_t, x_t)   # we only need z_src
    for w in CFG_SWEEP:
        z_pred = diffusion.ddim_sample(unet, z_src, dt_b,
                                       n_steps=DDIM_STEPS, cfg_scale=w)
        per_w[w] = decode_latent(z_pred).clamp(0, 1).cpu().numpy()

# Pairwise SSIM across scales (per slice, then aggregated)
rows = []
for w1, w2 in combinations(CFG_SWEEP, 2):
    a = per_w[w1]; b = per_w[w2]
    ssims = [ssim_fn(a[i, 0], b[i, 0], data_range=1.0) for i in range(N_CASES)]
    rows.append({'w1': w1, 'w2': w2,
                 'ssim_mean': float(np.mean(ssims)),
                 'ssim_min':  float(np.min(ssims))})
diag = pd.DataFrame(rows)
print(diag.to_string(index=False))

min_pairwise = diag['ssim_min'].min()
verdict = ('COLLAPSE'  if min_pairwise >= 1 - EPSILON
           else 'OK (conditioning has measurable effect)')
print(f'\nmin pairwise SSIM across w∈{CFG_SWEEP}: {min_pairwise:.4f}')
print(f'Definition 1 verdict (ε={EPSILON}): {verdict}')
diag.to_csv(OUT_DIR / 'cfg_collapse_diagnostic.csv', index=False)